# Pre-processing News Data

## Bài toán
Dữ liệu gồm n văn bản phân vào 10 chủ đề khác nhau. Cần biễu diễn mỗi văn bản dưới dạng một vector số thể hiện cho nội dụng của văn bản đó.

## Mục lục
- Load dữ liệu từ thư mục
- Loại bỏ các stop words
- Sử dụng thư viện để mã hóa TF-IDF cho mỗi văn bản

## Phương pháp mã hóa: TF-IDF
Cho tập gồm $n$ văn bản: $D = \{d_1, d_2, ... d_n\}$. Tập từ điển tương ứng được xây dựng từ $n$ văn bản này có độ dài là $m$
- Xét văn bản $d$ có $|d|$ từ và $t$ là một từ trong $d$. Mã hóa tf-idf của $t$ trong văn bản $d$ được biểu diễn:
\begin{equation}
    \begin{split}
        \text{tf}_{t, d} &= \frac{f_t}{|d|} \\
        \text{idf}_{t, d} &= \log\frac{n}{n_t}, \ \ \ \ n_t = |\{d\in D: t\in d\}| \\
        \text{tf-idf}_{t d} &= \text{tf}_{t, d} \times \text{idf}_{t, d}
    \end{split}
\end{equation}

- Khi đó văn bản $d$ được mã hóa là một vector $m$ chiều. Các từ xuất hiện trong d sẽ được thay bằng giá trị tf-idf tương ứng. Các từ không xuất hiện trong $d$ thì thay là 0

In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import load_files
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

%matplotlib inline

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyvi\ViTokenizer.py:24: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  model = pickle.load(fin)


## Load dữ liệu từ thư mục

Cấu trúc thư mục như sau 

- data/news_vnexpress/

    - Kinh tế: 
        - bài báo 1.txt 
        - bài báo 2.txt 
    - Pháp luật
        - bài báo 3.txt 
        - bài báo 4.txt 

In [3]:
INPUT = 'news_vnexpress'
os.makedirs("images",exist_ok=True)  # thư mục lưu các các hình ảnh trong quá trình huấn luyện và đánh gía

In [5]:
# statistics
print('Các nhãn và số văn bản tương ứng trong dữ liệu')
print('----------------------------------------------')
n = 0
for label in os.listdir(INPUT):
    print(f'{label}: {len(os.listdir(os.path.join(INPUT, label)))}')
    n += len(os.listdir(os.path.join(INPUT, label)))

print('-------------------------')
print(f"Tổng số văn bản: {n}")

Các nhãn và số văn bản tương ứng trong dữ liệu
----------------------------------------------
doi-song: 120
du-lich: 54
giai-tri: 201
giao-duc: 105
khoa-hoc: 144
kinh-doanh: 262
phap-luat: 59
suc-khoe: 162
the-thao: 173
thoi-su: 59
-------------------------
Tổng số văn bản: 1339


In [6]:
# load data
data_train = load_files(container_path=INPUT, encoding="utf-8")
print('mapping:')
for i in range(len(data_train.target_names)):
    print(f'{data_train.target_names[i]} - {i}')

print('--------------------------')
print(data_train.filenames[0:1])
# print(data_train.data[0:1])
print(data_train.target[0:1])
print(data_train.data[0:1])

print("\nTổng số  văn bản: {}" .format( len(data_train.filenames)))

mapping:
doi-song - 0
du-lich - 1
giai-tri - 2
giao-duc - 3
khoa-hoc - 4
kinh-doanh - 5
phap-luat - 6
suc-khoe - 7
the-thao - 8
thoi-su - 9
--------------------------
['news_vnexpress\\khoa-hoc\\00133.txt']
[4]
['Mời độc giả đặt câu hỏi tại đây\n']

Tổng số  văn bản: 1339


## Chuyển dữ liệu dạng text về ma trận (n x m) bằng TF-IDF

* Bạn cần viết đoạn mã tương ứng trong cell bên dưới. Theo các bước được gợi ý

In [9]:
# load dữ liệu các stopwords
stopwords_path = os.path.join('vietnamese-stopwords.txt')
if os.path.exists(stopwords_path):
    with open(stopwords_path, encoding='utf-8') as f:
        stopwords = [line.strip() for line in f if line.strip()]
else:
    stopwords = []
    print(f'Warning: stopwords file not found at {stopwords_path}. Proceeding without stopwords.')

# Chuyển hoá dữ liệu text về dạng vector TF 
#     - loại bỏ từ dừng
#     - sinh từ điển

def vietnamese_tokenizer(text):
    text = text.lower()
    return ViTokenizer.tokenize(text).split()

module_count_vector = CountVectorizer(
    tokenizer=vietnamese_tokenizer,
    stop_words=stopwords,
    lowercase=False,
    token_pattern=None,
)

module_tf_idf = TfidfTransformer()

# fit_transform trên toàn bộ dữ liệu train
X_tf = module_count_vector.fit_transform(data_train.data)
X = module_tf_idf.fit_transform(X_tf)

# Hàm thực hiện chuyển đổi dữ liệu text thành dữ liệu số dạng ma trận 
# Input: Dữ liệu 2 chiều dạng numpy.array, mảng nhãn id dạng numpy.array 

data_preprocessed = X

Y = data_train.target #nhan

print(f"\nSố lượng từ trong từ điển: {len(module_count_vector.vocabulary_)}")
print(f"Kích thước dữ liệu sau khi xử lý: {X.shape}")
print(f"Kích thước nhãn tương ứng: {Y.shape}")

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['a_ha', 'a_lô', 'ai_ai', 'ai_nấy', 'ba_ba', 'ba_cùng', 'bao', 'bao_giờ', 'bao_lâu', 'bao_nhiêu', 'bay_biến', 'biết_bao', 'biết_bao_nhiêu', 'biết_mấy', 'biết_đâu', 'bài_bác', 'bây', 'bây_chừ', 'bây_giờ', 'bây_nhiêu', 'bên_bị', 'bước_đi', 'bản_thân', 'bất', 'bất_chợt', 'bất_cứ', 'bất_giác', 'bất_kì', 'bất_kể', 'bất_kỳ', 'bất_luận', 'bất_ngờ', 'bất_nhược', 'bất_quá', 'bất_tử', 'bất_ý', 'bất_đồ', 'bấy_giờ', 'bấy_lâu', 'bấy_lâu_nay', 'bấy_nay', 'bấy_nhiêu', 'bẩy', 'bập', 'bập_bõm', 'bắt_đầu', 'bắt_đầu_từ', 'bằng_cứ', 'bằng_không', 'bị_chú', 'bỏ_bà', 'bỏ_cha', 'bỏ_cuộc', 'bỏ_mình', 'bỏ_mẹ', 'bỏ_nhỏ', 'bỏ_quá', 'bỗng_chốc', 'bỗng_dưng', 'bỗng_không', 'bỗng_nhiên', 'bỗng_đâu', 'bội_phần', 'bởi_thế', 'b


Số lượng từ trong từ điển: 22788
Kích thước dữ liệu sau khi xử lý: (1339, 22788)
Kích thước nhãn tương ứng: (1339,)


In [10]:
print(X[100].toarray())
print(Y[100])

[[0.         0.10448759 0.         ... 0.         0.         0.        ]]
5


In [11]:
sum(sum(X[100].toarray() != 0))

np.int64(261)

In [12]:
print(X[100])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 261 stored elements and shape (1, 22788)>
  Coords	Values
  (0, 1)	0.10448758854429772
  (0, 3)	0.08288923347146235
  (0, 6)	0.031841763405348555
  (0, 7)	0.031816922209168465
  (0, 10)	0.37593140878841935
  (0, 11)	0.024799138693026624
  (0, 12)	0.24958597216786366
  (0, 14)	0.028036690358435496
  (0, 103)	0.012151625492161118
  (0, 177)	0.03472014733019569
  (0, 234)	0.013700271319137446
  (0, 270)	0.018663041698347652
  (0, 381)	0.02139415584314372
  (0, 445)	0.04314556941581322
  (0, 563)	0.029446986225093633
  (0, 776)	0.030051972959316754
  (0, 798)	0.04294354510383223
  (0, 1007)	0.0348398237700963
  (0, 1214)	0.014441637746497269
  (0, 1250)	0.020421703904851696
  (0, 1357)	0.04249456634840991
  (0, 1656)	0.031142916345693956
  (0, 2002)	0.044920716529856734
  (0, 2017)	0.08984143305971347
  (0, 2027)	0.044920716529856734
  :	:
  (0, 21085)	0.02519967259642311
  (0, 21098)	0.027625822777869933
  (0, 21106)	0.14165296